# **Customer Support Arabic language Robot**


## Import Libraries

In [1]:
!pip install -q faiss-cpu sentence-transformers openai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 34.9 MB/s eta 0:00:00


In [2]:
pip install python-dotenv

In [14]:
import json
import os
from pathlib import Path
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from dotenv import load_dotenv
import time

## Auto-generate the knowledge base with an LLM

In [4]:
BASE_DIR = Path("/content")
DATA_DIR = BASE_DIR / "data"
INDEX_DIR = BASE_DIR / "index"
ENV_DIR = BASE_DIR / "env"
DATA_DIR.mkdir(exist_ok=True)
INDEX_DIR.mkdir(exist_ok=True)
ENV_DIR.mkdir(exist_ok=True)

In [8]:
FAQ_PATH = DATA_DIR / "faq_knowledge_base.json"
with open(FAQ_PATH, "r", encoding="utf-8") as f:
    faqs = json.load(f)

## Build the FAISS retrieval index

In [10]:
EMBEDDING_MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
texts = [f"{item['question']} {item['answer']}" for item in faqs]

In [11]:
embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)
embeddings = embedder.encode(texts, convert_to_numpy=True, normalize_embeddings=True)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [12]:
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings.astype(np.float32))
faiss.write_index(index, str(INDEX_DIR / "faq.index"))
with open(INDEX_DIR / "faq_metadata.json", "w", encoding="utf-8") as f:
    json.dump(faqs, f, ensure_ascii=False, indent=2)

## Setting up an API key

In [13]:
ENV_FILE = ENV_DIR / ".env"
base_url = "https://api.gapapi.com/v1"
model_name = "gpt-4o"

In [16]:
load_dotenv(ENV_FILE)
api_key = os.getenv("API_KEY")
os.environ["LLM_API_KEY"] = api_key
llm_client = OpenAI(api_key=api_key, base_url=base_url)

## Create Robot With RAG

In [17]:
TOP_K = 3
SIMILARITY_THRESHOLD = 0.45

In [18]:
HANDOFF_MESSAGE = (
    "لست متأكداً تماماً من الإجابة على هذا السؤال بناءً على المعلومات المتوفرة لدي. "
    "لا أريد أن أعطيك معلومات غير دقيقة، لذا سأقوم بتحويلك إلى أحد ممثلي خدمة العملاء -- "
    "يمكنك التواصل معهم عبر support@example.com أو بكتابة 'أريد التحدث مع موظف'."
)

In [19]:
class FAQChatbot:
    def __init__(self, faqs, index, embedder):
        self.faqs = faqs
        self.index = index
        self.embedder = embedder
        self.use_llm = bool(os.environ.get("LLM_API_KEY"))
        if self.use_llm:
            self.llm_client = OpenAI(api_key=os.environ["LLM_API_KEY"], base_url=base_url)
            self.model_name = model_name
        self.history = []
    def retrieve(self, query: str, k: int = TOP_K):
        query_vec = self.embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
        scores, indices = self.index.search(query_vec.astype(np.float32), k)
        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx == -1:
                continue
            results.append({"score": float(score), **self.faqs[idx]})
        return results
    def generate_answer(self, user_question: str, retrieved: list) -> str:
        context_block = "\n\n".join(f"Q: {r['question']}\nA: {r['answer']}" for r in retrieved)
        if self.use_llm:
            system_prompt = (
                "أنت مساعد خدمة عملاء ودود ومختصر. أجب على سؤال العميل "
                "باستخدام المعلومات الموجودة في سياق الأسئلة الشائعة أدناه فقط. "
                "إذا كان السياق لا يجيب على السؤال بشكل كامل، قل ذلك بصراحة بدلاً "
                "من التخمين. أجب دائماً باللغة العربية الفصحى، وبإيجاز."
            )
            user_prompt = f"سؤال العميل: {user_question}\n\nسياق الأسئلة الشائعة:\n{context_block}"
            response = self.llm_client.chat.completions.create(
                model=self.model_name,
                messages=[
                    {"role": "system", "content": system_prompt},
                    *self.history[-6:],
                    {"role": "user", "content": user_prompt},
                ],
                temperature=0.2,
            )
            answer = response.choices[0].message.content
        else:
            answer = retrieved[0]["answer"]
        self.history.append({"role": "user", "content": user_question})
        self.history.append({"role": "assistant", "content": answer})
        return answer
    def ask(self, user_question: str) -> str:
        retrieved = self.retrieve(user_question)
        if not retrieved or retrieved[0]["score"] < SIMILARITY_THRESHOLD:
            self.history.append({"role": "user", "content": user_question})
            self.history.append({"role": "assistant", "content": HANDOFF_MESSAGE})
            return HANDOFF_MESSAGE
        return self.generate_answer(user_question, retrieved)

In [20]:
bot = FAQChatbot(faqs, index, embedder)

## Chatting with the Bot


In [24]:
def bot_say(text, delay=3):
    tex = f"Mujeeb AI: {text}"
    if delay:
        time.sleep(delay)
    return tex

In [26]:
sentenses = [
    " مرحباً بك! أنا مساعد خدمة العملاء الخاص بك.",
    " يمكنك سؤالي عن الشحن، الإرجاع، الدفع، حسابك، طلباتك، أو الضمان.",
    " اكتب 'خروج' لإنهاء المحادثة.\n"
]

In [27]:
for sen in sentenses:
  print(bot_say(sen, delay=3))

while True:
    user_input = input("You: ").strip()
    if user_input.lower() in {"خروج"}:
        print("Mujeeb AI:  شكراً لتواصلك معنا، نتمنى لك يوماً سعيداً")
        break
    if not user_input:
        continue
    answer = bot.ask(user_input)
    print(f"Mujeeb AI: {answer}\n")


Mujeeb AI:  مرحباً بك! أنا مساعد خدمة العملاء الخاص بك.
Mujeeb AI:  يمكنك سؤالي عن الشحن، الإرجاع، الدفع، حسابك، طلباتك، أو الضمان.
Mujeeb AI:  اكتب 'خروج' لإنهاء المحادثة.

You: هل المنتجات الإلكترونية مشمولة بالضمان؟
Mujeeb AI: نعم، معظم المنتجات الإلكترونية مشمولة بضمان الشركة المصنعة لمدة 12 شهراً ضد العيوب المصنعية.

You: خروج
Mujeeb AI:  شكراً لتواصلك معنا، نتمنى لك يوماً سعيداً
